<a href="https://colab.research.google.com/github/HyunjuYun1009/tda-conference/blob/main/pdgnn/hetero-gnn/pdgnn_metapath_dblp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. zip 풀기
!unzip -o /content/hetero_pdgnn.zip -d /content/
!ls /content/hetero_pdgnn/

Archive:  /content/hetero_pdgnn.zip
   creating: /content/hetero_pdgnn/
  inflating: /content/hetero_pdgnn/epd_features.py  
  inflating: /content/hetero_pdgnn/models.py  
 extracting: /content/hetero_pdgnn/__init__.py  
  inflating: /content/hetero_pdgnn/train.py  
  inflating: /content/hetero_pdgnn/run_colab.py  
  inflating: /content/hetero_pdgnn/data_metapath.py  
  inflating: /content/hetero_pdgnn/colab_setup.py  
colab_setup.py	  epd_features.py  models.py	 train.py
data_metapath.py  __init__.py	   run_colab.py


In [2]:
# 2. TLC-GNN clone + 의존성 설치
!git clone https://github.com/pkuyzy/TLC-GNN.git

import torch
v = torch.__version__.split('+')[0]
cuda = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{v}+{cuda}.html -q
!pip install torch-geometric gudhi networkx scikit-learn -q

Cloning into 'TLC-GNN'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 95 (delta 36), reused 77 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 5.47 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (36/36), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 37.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 134.8 MB/s eta 0:00:00


In [3]:
# 3. 패치 적용 + sg2dgm 빌드
import shutil
shutil.copy('/content/dgformat.py',      '/content/TLC-GNN/sg2dgm/dgformat.py')
shutil.copy('/content/riccidist2dgm.py', '/content/TLC-GNN/sg2dgm/riccidist2dgm.py')

%cd /content/TLC-GNN/sg2dgm
!python setup_PI.py build_ext --inplace || cp persistenceImager.pyx persistenceImager.py
%cd /content
print('setup done')

/content/TLC-GNN/sg2dgm
Compiling PersistenceImager.pyx because it changed.
[1/1] Cythonizing PersistenceImager.pyx
/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/TLC-GNN/sg2dgm/PersistenceImager.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
/content
setup done


In [4]:
import sys, os, time
sys.path.insert(0, "/content/TLC-GNN")
sys.path.insert(0, "/content")

import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

from hetero_pdgnn.data_metapath import (
    load_dblp, build_author_subgraphs, add_metapath_edges_to_data,
    AUTHOR_METAPATHS,
)
from hetero_pdgnn.epd_features import build_metapath_epd_features
from hetero_pdgnn.train import compare

device: cuda


In [5]:
data = load_dblp(root="/content/dblp_data")
print(data)
print("classes:", int(data["author"].y.max().item()) + 1)
print("train/val/test:",
      int(data["author"].train_mask.sum()),
      int(data["author"].val_mask.sum()),
      int(data["author"].test_mask.sum()))

Extracting /content/dblp_data/raw/DBLP_processed.zip
Processing...


HeteroData(
  author={
    x=[4057, 334],
    y=[4057],
    train_mask=[4057],
    val_mask=[4057],
    test_mask=[4057],
  },
  paper={ x=[14328, 4231] },
  term={ x=[7723, 50] },
  conference={
    num_nodes=20,
    x=[20, 20],
  },
  (author, to, paper)={ edge_index=[2, 19645] },
  (paper, to, author)={ edge_index=[2, 19645] },
  (paper, to, term)={ edge_index=[2, 85810] },
  (paper, to, conference)={ edge_index=[2, 14328] },
  (term, to, paper)={ edge_index=[2, 85810] },
  (conference, to, paper)={ edge_index=[2, 14328] }
)
classes: 4
train/val/test: 400 400 3257


Done!


In [6]:
data = add_metapath_edges_to_data(data)
for et in data.edge_types:
    if et[0] == "author" and et[2] == "author":
        print(f"  {et}: {data[et].edge_index.shape[1]} edges")

  ('author', 'APA', 'author'): 7056 edges
  ('author', 'APCPA', 'author'): 4996438 edges
  ('author', 'APTPA', 'author'): 7039514 edges


In [7]:
subs = build_author_subgraphs(data)
for name, G in subs.items():
    print(f"{name:6s}: |V|={G.number_of_nodes()}  |E|={G.number_of_edges()}")

APA   : |V|=4057  |E|=3528
APCPA : |V|=4057  |E|=2498219
APTPA : |V|=4057  |E|=3519757


In [8]:
import scipy.sparse as sp
import numpy as np
from hetero_pdgnn import data_metapath as dm

# threshold-aware version
_orig_metapath_adjacency = dm.metapath_adjacency

def metapath_adjacency_threshold(data, metapath, min_count=1):
    """Keep only entries with count >= min_count."""
    A = _orig_metapath_adjacency(data, metapath)
    A.data = (A.data >= min_count).astype(np.uint8)
    A.eliminate_zeros()
    return A

# APA는 그대로(1), APCPA/APTPA는 더 강한 threshold
THRESHOLDS = {"APA": 1, "APCPA": 3, "APTPA": 2}

def metapath_subgraph_nx_thr(data, metapath, name):
    A = metapath_adjacency_threshold(data, metapath, THRESHOLDS[name])
    A = (A > 0).astype(np.uint8)
    A = A.maximum(A.T)
    A = A.tolil(); A.setdiag(0); A = A.tocsr(); A.eliminate_zeros()
    import networkx as nx
    n = data[metapath[0][0]].num_nodes
    G = nx.Graph(); G.add_nodes_from(range(n))
    coo = A.tocoo()
    mask = coo.row < coo.col
    G.add_edges_from(zip(coo.row[mask].tolist(), coo.col[mask].tolist()))
    return G

# subgraph 재빌드
subs = {name: metapath_subgraph_nx_thr(data, mp, name)
        for name, mp in dm.AUTHOR_METAPATHS.items()}
for name, G in subs.items():
    print(f"{name:6s}: |V|={G.number_of_nodes()}  |E|={G.number_of_edges()}")

APA   : |V|=4057  |E|=3528
APCPA : |V|=4057  |E|=1181170
APTPA : |V|=4057  |E|=2255911


In [9]:
import numpy as np
from hetero_pdgnn import data_metapath as dm

for name, mp in dm.AUTHOR_METAPATHS.items():
    A = dm._orig_metapath_adjacency(data, mp) if hasattr(dm, '_orig_metapath_adjacency') else dm.metapath_adjacency(data, mp)
    counts = A.data
    print(f"{name:6s}: nnz={A.nnz:>9d}  "
          f"min={counts.min()}  median={int(np.median(counts))}  "
          f"p90={int(np.percentile(counts,90))}  p99={int(np.percentile(counts,99))}  max={counts.max()}")

APA   : nnz=    11113  min=1.0  median=1  p90=6  p99=22  max=168.0
APCPA : nnz=  5000495  min=1.0  median=2  p90=12  p99=61  max=4124.0
APTPA : nnz=  7043571  min=1.0  median=2  p90=11  p99=60  max=14461.0


In [10]:
import scipy.sparse as sp
import numpy as np
import networkx as nx
from hetero_pdgnn import data_metapath as dm

def knn_sparsify(A, k):
    """각 행에서 top-k 값만 유지 (대각 제외). vectorized."""
    A = A.tocsr().copy()
    A.setdiag(0)
    A.eliminate_zeros()

    new_data, new_indices, new_indptr = [], [], [0]
    for i in range(A.shape[0]):
        s, e = A.indptr[i], A.indptr[i+1]
        vals  = A.data[s:e]
        cols  = A.indices[s:e]
        if len(vals) <= k:
            new_data.append(vals); new_indices.append(cols)
        else:
            top = np.argpartition(-vals, k)[:k]
            new_data.append(vals[top]); new_indices.append(cols[top])
        new_indptr.append(new_indptr[-1] + (len(new_data[-1])))
    return sp.csr_matrix(
        (np.concatenate(new_data),
         np.concatenate(new_indices),
         np.array(new_indptr)),
        shape=A.shape,
    )

def metapath_subgraph_knn(data, metapath, k):
    A = dm.metapath_adjacency(data, metapath)
    A = knn_sparsify(A, k=k)
    A = (A > 0).astype(np.uint8)
    A = A.maximum(A.T)                 # 대칭화
    A = A.tolil(); A.setdiag(0); A = A.tocsr(); A.eliminate_zeros()
    n = data[metapath[0][0]].num_nodes
    G = nx.Graph(); G.add_nodes_from(range(n))
    coo = A.tocoo()
    mask = coo.row < coo.col
    G.add_edges_from(zip(coo.row[mask].tolist(), coo.col[mask].tolist()))
    return G

# APA는 sparse하니 k 안 적용해도 무방하지만 일관성 위해 동일 k 사용
K = 20
subs = {name: metapath_subgraph_knn(data, mp, K)
        for name, mp in dm.AUTHOR_METAPATHS.items()}

for name, G in subs.items():
    n, m = G.number_of_nodes(), G.number_of_edges()
    iso = sum(1 for v in G.nodes if G.degree(v) == 0)
    print(f"{name:6s}: |V|={n}  |E|={m:7d}  avg_deg={2*m/n:5.1f}  isolated={iso}")

APA   : |V|=4057  |E|=   3524  avg_deg=  1.7  isolated=1466
APCPA : |V|=4057  |E|=  80264  avg_deg= 39.6  isolated=0
APTPA : |V|=4057  |E|=  80486  avg_deg= 39.7  isolated=0


In [14]:
!rm -f /content/epd_cache/epd_APCPA_*.npy /content/epd_cache/epd_APTPA_*.npy
!ls /content/epd_cache/

epd_APA_035da7044d22_hop2_res5.npy  epd_APA_cd1579100aa1_hop2_res5.npy


In [11]:
import time
t0 = time.time()
epd_np = build_metapath_epd_features(
    subs,
    hop=1,                       # ← 2에서 1로 변경
    resolution=5,
    cache_dir="/content/epd_cache",
    verbose=True,
)
print(f"EPD features shape: {epd_np.shape}   elapsed {time.time()-t0:.1f}s")
epd_feat = torch.from_numpy(epd_np)

[APA] computing EPD on 4057 nodes...
  [500/4057] elapsed 0.2s
  [1000/4057] elapsed 0.4s
  [1500/4057] elapsed 0.5s
  [2000/4057] elapsed 0.7s
  [2500/4057] elapsed 0.9s
  [3000/4057] elapsed 1.1s
  [3500/4057] elapsed 1.2s
  [4000/4057] elapsed 1.3s
[APCPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 10.9s
  [1000/4057] elapsed 16.4s
  [1500/4057] elapsed 24.4s
  [2000/4057] elapsed 29.5s
  [2500/4057] elapsed 35.4s
  [3000/4057] elapsed 38.9s
  [3500/4057] elapsed 42.3s
  [4000/4057] elapsed 45.6s
[APTPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 8.0s
  [1000/4057] elapsed 13.6s
  [1500/4057] elapsed 23.3s
  [2000/4057] elapsed 29.0s
  [2500/4057] elapsed 36.1s
  [3000/4057] elapsed 40.3s
  [3500/4057] elapsed 44.1s
  [4000/4057] elapsed 47.8s
EPD features shape: (4057, 75)   elapsed 95.6s


In [16]:
# 처음 실행 cell 6
from torch_geometric.data import HeteroData

keep_types = [et for et in data.edge_types
              if et[0] == "author" and et[2] == "author" and et[1] in AUTHOR_METAPATHS]
print("HAN will use edge types:", keep_types)

slim = HeteroData()
for nt in data.node_types:
    slim[nt].x = data[nt].x
slim["author"].y = data["author"].y
slim["author"].train_mask = data["author"].train_mask
slim["author"].val_mask   = data["author"].val_mask
slim["author"].test_mask  = data["author"].test_mask
for et in keep_types:
    slim[et].edge_index = data[et].edge_index
print(slim)

HAN will use edge types: [('author', 'APA', 'author'), ('author', 'APCPA', 'author'), ('author', 'APTPA', 'author')]
HeteroData(
  author={
    x=[4057, 334],
    y=[4057],
    train_mask=[4057],
    val_mask=[4057],
    test_mask=[4057],
  },
  paper={ x=[14328, 4231] },
  term={ x=[7723, 50] },
  conference={ x=[20, 20] },
  (author, APA, author)={ edge_index=[2, 7056] },
  (author, APCPA, author)={ edge_index=[2, 4996438] },
  (author, APTPA, author)={ edge_index=[2, 7039514] }
)


In [12]:
# 두번째 실행 cell 6
import torch
from torch_geometric.data import HeteroData

slim = HeteroData()
for nt in data.node_types:
    slim[nt].x = data[nt].x
slim["author"].y = data["author"].y
slim["author"].train_mask = data["author"].train_mask
slim["author"].val_mask   = data["author"].val_mask
slim["author"].test_mask  = data["author"].test_mask

# k-NN sparsified subgraph 엣지 사용 (HAN과 EPD가 같은 그래프 보게)
for name, G in subs.items():
    edges = list(G.edges())
    if edges:
        ei = torch.tensor(edges, dtype=torch.long).t().contiguous()
        ei = torch.cat([ei, ei.flip(0)], dim=1)   # 양방향
    else:
        ei = torch.zeros((2, 0), dtype=torch.long)
    slim[("author", name, "author")].edge_index = ei

for et in slim.edge_types:
    print(et, slim[et].edge_index.shape[1])
print(slim)

('author', 'APA', 'author') 7048
('author', 'APCPA', 'author') 160528
('author', 'APTPA', 'author') 160972
HeteroData(
  author={
    x=[4057, 334],
    y=[4057],
    train_mask=[4057],
    val_mask=[4057],
    test_mask=[4057],
  },
  paper={ x=[14328, 4231] },
  term={ x=[7723, 50] },
  conference={ x=[20, 20] },
  (author, APA, author)={ edge_index=[2, 7048] },
  (author, APCPA, author)={ edge_index=[2, 160528] },
  (author, APTPA, author)={ edge_index=[2, 160972] }
)


In [19]:
results = compare(
    slim,
    epd_feat,
    hidden_channels=64,
    heads=8,
    seeds=[0, 1, 2, 3, 4],
    device=device,
    verbose=False,
)
print()
print("=" * 50)
print(f"{'Model':<12} {'Test F1 mean':>14} {'std':>8}")
print("-" * 50)
for name, r in results.items():
    print(f"{name:<12} {r['mean']:>14.4f} {r['std']:>8.4f}")
print("=" * 50)
print("Per-seed:", results)


Model          Test F1 mean      std
--------------------------------------------------
HAN                  0.9326   0.0047
HAN+EPD              0.9347   0.0015
Per-seed: {'HAN': {'mean': 0.9326260524458128, 'std': 0.004679235373637264, 'all': [0.9366972347169057, 0.9336773771297471, 0.9341978674034974, 0.9350633030644262, 0.9234944799144879]}, 'HAN+EPD': {'mean': 0.9347101368037911, 'std': 0.0015023375966747832, 'all': [0.9332329871830625, 0.9326464456613304, 0.9353115560888782, 0.9364648867234666, 0.9358948083622187]}}


In [13]:
# 시드 10개로 늘리기
results = compare(
    slim, epd_feat,
    hidden_channels=64, heads=8,
    seeds=[0,1,2,3,4,5,6,7,8,9],
    device=device,
    verbose=False,
)
print()
for name, r in results.items():
    print(f"{name:<10}  mean {r['mean']:.4f}  std {r['std']:.4f}")
print("Per-seed:", results)


HAN         mean 0.9332  std 0.0037
HAN+EPD     mean 0.9344  std 0.0020
Per-seed: {'HAN': {'mean': 0.9332143892341088, 'std': 0.0037215039598947654, 'all': [0.9364938165718617, 0.9336773771297471, 0.9361529551868614, 0.9350633030644262, 0.9234944799144879, 0.9312120679896988, 0.9325025923246595, 0.9319230950948979, 0.9366235863180331, 0.935000618746414]}, 'HAN+EPD': {'mean': 0.9344359635354975, 'std': 0.0019932286857833725, 'all': [0.9332329871830625, 0.9326464456613304, 0.9353115560888782, 0.9364648867234666, 0.9362597157524266, 0.9321186703561054, 0.9352909479362522, 0.9355974121495461, 0.930686055723455, 0.93675095778045]}}


In [14]:
# 1) res=5 캐시 비우기
!rm -f /content/epd_cache/epd_*.npy
!ls /content/epd_cache/



In [15]:
# 2) EPD 재계산 (res 5 -> 10, hop 그대로 1)
import time
t0 = time.time()
epd_np = build_metapath_epd_features(
    subs,
    hop=1,
    resolution=10,              # ← 핵심 변경
    cache_dir="/content/epd_cache",
    verbose=True,
)
print(f"EPD shape: {epd_np.shape}  ({time.time()-t0:.1f}s)")
epd_feat = torch.from_numpy(epd_np)

[APA] computing EPD on 4057 nodes...
  [500/4057] elapsed 0.2s
  [1000/4057] elapsed 0.4s
  [1500/4057] elapsed 0.5s
  [2000/4057] elapsed 0.7s
  [2500/4057] elapsed 0.9s
  [3000/4057] elapsed 1.1s
  [3500/4057] elapsed 1.2s
  [4000/4057] elapsed 1.4s
[APCPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 9.2s
  [1000/4057] elapsed 15.3s
  [1500/4057] elapsed 22.9s
  [2000/4057] elapsed 28.1s
  [2500/4057] elapsed 33.9s
  [3000/4057] elapsed 38.1s
  [3500/4057] elapsed 41.6s
  [4000/4057] elapsed 44.9s
[APTPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 7.6s
  [1000/4057] elapsed 13.2s
  [1500/4057] elapsed 23.1s
  [2000/4057] elapsed 28.8s
  [2500/4057] elapsed 36.6s
  [3000/4057] elapsed 40.8s
  [3500/4057] elapsed 44.7s
  [4000/4057] elapsed 48.4s
EPD shape: (4057, 300)  (95.5s)


In [16]:
# 3) 10 seed 다시 비교 (Cell 6의 slim은 그대로)
results = compare(
    slim, epd_feat,
    hidden_channels=64, heads=8,
    seeds=list(range(10)),
    device=device,
    verbose=False,
)
print()
for name, r in results.items():
    print(f"{name:<10}  mean {r['mean']:.4f}  std {r['std']:.4f}")
print("Per-seed:", results)


HAN         mean 0.9332  std 0.0037
HAN+EPD     mean 0.9338  std 0.0024
Per-seed: {'HAN': {'mean': 0.9332199811682342, 'std': 0.0037196009849890534, 'all': [0.9364938165718617, 0.9336773771297471, 0.9361529551868614, 0.9350633030644262, 0.9234944799144879, 0.9312120679896988, 0.9325025923246595, 0.931979014436153, 0.9366235863180331, 0.935000618746414]}, 'HAN+EPD': {'mean': 0.93377041471065, 'std': 0.00240085233600516, 'all': [0.9344184785062716, 0.9323898451325994, 0.9346031594112237, 0.9305352378028161, 0.935595043263769, 0.9331372100906778, 0.9301654334681051, 0.9335414768894816, 0.9343475040812762, 0.9389707584602787]}}


In [17]:
# epd_features.py의 _assign_distance_filter 대신 degree 기반 사용
import networkx as nx, numpy as np

def _assign_degree_filter(G_ego, source, key="min"):
    """source 무관, ego 내 degree를 normalized filter로."""
    degs = dict(G_ego.degree())
    max_d = max(degs.values()) or 1
    for n in G_ego.nodes():
        G_ego.nodes[n][key] = float(degs[n]) / max_d
    for u, v in G_ego.edges():
        G_ego.edges[u, v][key] = max(G_ego.nodes[u][key], G_ego.nodes[v][key])

# 함수 교체
from hetero_pdgnn import epd_features as ef
ef._assign_distance_filter = _assign_degree_filter

# 캐시 비우고 res=5로 재계산
!rm -f /content/epd_cache/epd_*.npy

import time, torch
t0 = time.time()
epd_np = ef.build_metapath_epd_features(
    subs, hop=1, resolution=5,
    cache_dir="/content/epd_cache", verbose=True,
)
print(f"EPD shape: {epd_np.shape}  ({time.time()-t0:.1f}s)")
epd_feat = torch.from_numpy(epd_np)

# 10 seed 다시
results = compare(slim, epd_feat,
    hidden_channels=64, heads=8,
    seeds=list(range(10)), device=device, verbose=False)
print()
for name, r in results.items():
    print(f"{name:<10}  mean {r['mean']:.4f}  std {r['std']:.4f}")

[APA] computing EPD on 4057 nodes...
  [500/4057] elapsed 0.2s
  [1000/4057] elapsed 0.3s
  [1500/4057] elapsed 0.5s
  [2000/4057] elapsed 0.7s
  [2500/4057] elapsed 0.9s
  [3000/4057] elapsed 1.0s
  [3500/4057] elapsed 1.1s
  [4000/4057] elapsed 1.2s
[APCPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 22.3s
  [1000/4057] elapsed 34.0s
  [1500/4057] elapsed 52.3s
  [2000/4057] elapsed 62.8s
  [2500/4057] elapsed 74.9s
  [3000/4057] elapsed 80.0s
  [3500/4057] elapsed 84.6s
  [4000/4057] elapsed 88.9s
[APTPA] computing EPD on 4057 nodes...
  [500/4057] elapsed 17.1s
  [1000/4057] elapsed 28.6s
  [1500/4057] elapsed 53.2s
  [2000/4057] elapsed 64.0s
  [2500/4057] elapsed 79.9s
  [3000/4057] elapsed 85.6s
  [3500/4057] elapsed 91.1s
  [4000/4057] elapsed 96.0s
EPD shape: (4057, 75)  (187.3s)

HAN         mean 0.9332  std 0.0037
HAN+EPD     mean 0.9283  std 0.0020


In [18]:
for name, r in results.items():
    print(f"{name:<10}  mean {r['mean']:.4f}  std {r['std']:.4f}")
print("Per-seed:", results)

HAN         mean 0.9332  std 0.0037
HAN+EPD     mean 0.9283  std 0.0020
Per-seed: {'HAN': {'mean': 0.9331602416245459, 'std': 0.0036875101341377887, 'all': [0.9366972347169057, 0.9336895628928561, 0.9353958751830798, 0.9350633030644262, 0.9234944799144879, 0.9312120679896988, 0.9325025923246595, 0.9319230950948979, 0.9366235863180331, 0.935000618746414]}, 'HAN+EPD': {'mean': 0.9283412705540218, 'std': 0.0020358932789207628, 'all': [0.9295852029900771, 0.9327257418402468, 0.9282959167877795, 0.9307012715208716, 0.9254694483168615, 0.9267564258887091, 0.9275063702466941, 0.9266073671303019, 0.927548980361361, 0.9282159804573151]}}
